# Lab 2 · PyTorch image classification — Independent assignment

**BME · MSc Deep Learning / VITMMA19 · Fall 2026/2027**  
Department of Telecommunications and Artificial Intelligence  
**Instructor:** Dr. Mohammed Salah Al-Radhi


**Five tasks · 20 points · about 45 minutes + 5 minutes to submit**

Apply the guided workflow to CIFAR-10: 32×32 RGB images in ten classes. Use the same core ideas with a different dataset and a slightly deeper CNN.
The classroom setting uses **6,000 training, 1,000 validation, and 1,000 held-out test images**, with **two epochs per experiment**. CPU is supported; a GPU is optional.
Do not change the split, seed, batch size, epoch budget, or architecture during the core comparison.

Use the guided notebook and official documentation as references. Work on your own submission, follow course rules for collaboration and AI tools, record assistance, and be ready to explain your code.

**Submit one executed `.ipynb` file to Moodle by the end of the lab.** Some cells intentionally stop at TODOs until you complete them; supplied helpers are labelled.


## Your details

**Name:** YOUR NAME  
**Neptun code:** YOUR NEPTUN CODE  
**Assistance used (if any):** YOUR ANSWER


## 0. Setup and data — provided

Save a copy in Drive. Run the setup and data cells first. Dataset downloads require internet access and may take longer than an epoch.
Use the installed Colab packages. Keep the chosen device for both experiments.
The seed controls our split and initialisation; minor numerical differences across hardware/package versions remain possible.


In [ ]:
import os
import random
import time
import platform
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
DATA_ROOT = 'data'  # Use the same cache directory for both notebooks.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(2, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    info = torch.cuda.get_device_properties(DEVICE)
    print('GPU:', info.name, '| total memory:', round(info.total_memory / 2**30, 2), 'GiB')
else:
    print('CPU mode: the core tasks do not require a GPU.')


In [ ]:
#@title Provided data helpers — run this cell; expand to inspect
def training_channel_stats(raw_images, train_indices):
    """Mean/std over selected training images and spatial positions, per channel."""
    total = None
    total_sq = None
    count = 0
    # Small chunks avoid making a floating-point copy of the full dataset.
    for start in range(0, len(train_indices), 256):
        idx = train_indices[start:start + 256]
        batch = np.asarray(raw_images[idx], dtype=np.float64) / 255.0
        if batch.ndim == 3:  # Fashion-MNIST: N,H,W -> N,H,W,1
            batch = batch[..., None]
        pixels = batch.reshape(-1, batch.shape[-1])
        if total is None:
            total = np.zeros(pixels.shape[1])
            total_sq = np.zeros(pixels.shape[1])
        total += pixels.sum(axis=0)
        total_sq += (pixels ** 2).sum(axis=0)
        count += pixels.shape[0]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-12))
    return tuple(mean.tolist()), tuple(std.tolist())

def make_loader(dataset, shuffle=False, seed=SEED):
    # A separate generator keeps shuffle order independent of plotting/model code.
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
                      generator=torch.Generator().manual_seed(seed))

def show_images(images, labels, class_names, mean, std, predictions=None, n=10):
    images = images.detach().cpu()
    n = min(n, len(images))
    if n == 0:
        print('No images to display.')
        return
    columns = min(5, n)
    rows = (n + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(2.6 * columns, 2.7 * rows), squeeze=False)
    m = torch.tensor(mean).view(-1, 1, 1)
    s = torch.tensor(std).view(-1, 1, 1)
    for i, ax in enumerate(axes.flat):
        ax.axis('off')
        if i >= n:
            continue
        restored = (images[i] * s + m).clamp(0, 1)
        if restored.shape[0] == 1:
            ax.imshow(restored[0], cmap='gray', vmin=0, vmax=1)
        else:
            ax.imshow(restored.permute(1, 2, 0))
        title = 'True: ' + class_names[int(labels[i])]
        if predictions is not None:
            title += '\nPred: ' + class_names[int(predictions[i])]
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    plt.show()


In [ ]:
BATCH_SIZE = 128
N_TRAIN, N_VAL, N_TEST = 6000, 1000, 1000
EPOCHS = 2

raw_train = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True)
permutation = torch.randperm(len(raw_train), generator=torch.Generator().manual_seed(SEED)).tolist()
train_idx = permutation[:N_TRAIN]
val_idx = permutation[N_TRAIN:N_TRAIN + N_VAL]
assert set(train_idx).isdisjoint(val_idx)

MEAN, STD = training_channel_stats(raw_train.data, train_idx)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
raw_train.transform = transform
train_dataset = Subset(raw_train, train_idx)
val_dataset = Subset(raw_train, val_idx)
class_names = raw_train.classes
train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset)

print('Training / validation:', len(train_dataset), '/', len(val_dataset))
print('Training-only channel mean:', np.round(MEAN, 4))
print('Training-only channel std :', np.round(STD, 4))
print('Epochs per experiment:', EPOCHS)
print('The test split is held back until final evaluation.')


The helper calculates per-channel statistics from **our 6,000 training images only**. Validation and test images reuse the same mean/std.
This implements the train-only standardisation rule from Lab 1 and yesterday's lecture.

The unused training examples and remaining test images are not part of this short classroom exercise. A score from this subset is not a full CIFAR-10 benchmark result.


## T1 · Inspect the data — 3 points · about 6 minutes

1. Obtain one training mini-batch. Print image/label shapes and dtypes, and the minimum/maximum normalised pixel values. **(1 point)**
2. Display ten images in a 2×5 grid. You may use the provided `show_images` helper, which undoes normalisation. **(1 point)**
3. Explain the channel dimension and why validation/test images use training statistics. **(1 point)**


In [ ]:
# TODO T1: obtain images and labels, print the requested values, then plot.
# Hint: next(iter(train_loader))
# Plot helper: show_images(images, labels, class_names, MEAN, STD, n=10)

# YOUR CODE HERE


### T1 · Your answer

What does the `3` mean in `[128, 3, 32, 32]`? Why must validation/test images use training mean/std?

YOUR ANSWER (2–3 sentences)


## T2 · Complete the CNN and inspect its memory — 5 points · about 7 minutes

Replace all eight `None` values. Each convolution has kernel size 3, stride 1, and padding 1, followed by ReLU and 2×2 max pooling.

| Stage | Required shape, excluding batch |
| --- | --- |
| Input | 3 × 32 × 32 |
| Block 1 | 32 × 16 × 16 |
| Block 2 | 64 × 8 × 8 |
| Block 3 | 128 × 4 × 4 |
| Classifier | Flatten → 128 hidden units → ReLU → dropout 0.30 → 10 logits |

Run the supplied shape and memory checks. Do not add softmax. Explain why the printed weight storage is smaller than the memory required for training.


In [ ]:
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        conv2_in = None
        conv2_out = None
        conv3_in = None
        conv3_out = None
        flattened_features = None
        hidden_units = None
        dropout_p = None
        num_classes = None
        values = [conv2_in, conv2_out, conv3_in, conv3_out,
                  flattened_features, hidden_units, dropout_p, num_classes]
        assert all(v is not None for v in values), 'T2: replace the None placeholders first.'
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(conv2_in, conv2_out, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(conv3_in, conv3_out, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(flattened_features, hidden_units), nn.ReLU(),
            nn.Dropout(dropout_p), nn.Linear(hidden_units, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

# The shape probe is separate from the fresh model trained in T3.
shape_model = CIFAR10CNN().to(DEVICE).eval()
with torch.no_grad():
    logits = shape_model(torch.zeros(4, 3, 32, 32, device=DEVICE))
print('Output shape:', tuple(logits.shape))
assert logits.shape == (4, 10), 'Expected one vector of 10 logits per image.'

parameter_count = sum(p.numel() for p in shape_model.parameters())
weight_bytes = sum(p.numel() * p.element_size() for p in shape_model.parameters())
print('Parameters:', parameter_count)
print('FP32 weights:', weight_bytes, 'bytes =', round(weight_bytes / 2**20, 3), 'MiB')


### T2 · Your answer

Explain the flatten size. Why is the printed weight-storage estimate not the full training-memory requirement?

YOUR ANSWER (2–3 sentences)


## T3 · Complete one training/evaluation cycle — 5 points · about 12 minutes

Complete the two functions below, then run the diagnostic and the supplied two-epoch baseline driver.

- Training: move images/labels to the device, clear old gradients, calculate logits and cross-entropy loss, call backward, and update parameters.
- Evaluation: move the batch and calculate logits/loss. Keep `model.eval()` and `@torch.no_grad()`; do not update weights.
- Answer the two questions below the curves.

The supplied metric accumulation weights batch losses by the number of images. Label targets are class indices, not one-hot vectors.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for images, labels in loader:
        # TODO: move both tensors to device.
        # TODO: clear gradients, compute logits and loss, backpropagate, then update.
        raise NotImplementedError('T3: complete the training step and remove this line.')
        logits = None
        loss = None
        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, total_correct / total_samples

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for images, labels in loader:
        # TODO: move the batch to device, then compute logits and loss.
        # Do not call backward() or optimizer.step() here.
        raise NotImplementedError('T3: complete evaluation and remove this line.')
        logits = None
        loss = None
        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, total_correct / total_samples


In [ ]:
#@title Quick diagnostic before the full run (not an automatic grade)
set_seed()
probe_model = CIFAR10CNN().to(DEVICE)
probe_images, probe_labels = next(iter(make_loader(train_dataset)))
probe_loader = DataLoader(TensorDataset(probe_images[:8], probe_labels[:8]), batch_size=8)
criterion_probe = nn.CrossEntropyLoss()
opt_probe = torch.optim.SGD(probe_model.parameters(), lr=0.01)
before = {k: v.detach().clone() for k, v in probe_model.state_dict().items()}
small_train_loss, _ = train_one_epoch(probe_model, probe_loader, criterion_probe, opt_probe, DEVICE)
assert np.isfinite(small_train_loss), 'Training loss is not finite.'
assert any(not torch.equal(before[k], v) for k, v in probe_model.state_dict().items()), 'No weights changed: check backward() and step().'
after_training = {k: v.detach().clone() for k, v in probe_model.state_dict().items()}
small_val_loss, small_val_acc = evaluate(probe_model, probe_loader, criterion_probe, DEVICE)
assert np.isfinite(small_val_loss) and 0 <= small_val_acc <= 1
assert all(torch.equal(after_training[k], v) for k, v in probe_model.state_dict().items()), 'Evaluation changed model state.'
assert not probe_model.training, 'Evaluation should put the model in eval mode.'
print('Diagnostic passed: training changes weights; evaluation preserves them.')


### Baseline training — provided

The driver resets the seed and constructs a fresh model/optimizer/loader for each run. It restores the checkpoint with the highest validation accuracy (earlier epoch wins ties).
The returned `baseline_run['model']` is therefore the **best validation checkpoint**, not simply the last epoch.


In [ ]:
#@title Provided experiment driver — records curves and restores the best checkpoint
def run_experiment(model_factory, learning_rate, epochs, train_data, val_data, run_name):
    # Each call starts from the same seed, fresh weights, fresh optimizer, and shuffle sequence.
    set_seed(SEED)
    model = model_factory().to(DEVICE)
    initial_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    train_loader_run = make_loader(train_data, shuffle=True)
    val_loader_run = make_loader(val_data)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'seconds': []}
    best_acc, best_epoch, best_state = -1.0, None, None
    for epoch in range(1, epochs + 1):
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        train_loss, train_acc = train_one_epoch(model, train_loader_run, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader_run, criterion, DEVICE)
        if not np.isfinite([train_loss, train_acc, val_loss, val_acc]).all():
            raise ValueError('Non-finite metric: inspect the loss, data, and learning rate.')
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        for key, value in [('train_loss', train_loss), ('train_acc', train_acc),
                           ('val_loss', val_loss), ('val_acc', val_acc), ('seconds', elapsed)]:
            history[key].append(value)
        if val_acc > best_acc:  # In a tie, keep the earlier checkpoint.
            best_acc, best_epoch = val_acc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'{run_name} | epoch {epoch}/{epochs} | '
              f'train loss {train_loss:.3f}, acc {train_acc:.3f} | '
              f'val loss {val_loss:.3f}, acc {val_acc:.3f} | {elapsed:.1f} s')
    model.load_state_dict(best_state)
    model.eval()
    print(f'{run_name}: restored epoch {best_epoch}, validation accuracy {best_acc:.3f}')
    return {'name': run_name, 'model': model, 'history': history, 'lr': learning_rate,
            'best_val_acc': best_acc, 'best_epoch': best_epoch, 'initial_state': initial_state}

def plot_history(history, title):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for ax, metric, label in [(axes[0], 'loss', 'Cross-entropy loss'), (axes[1], 'acc', 'Accuracy')]:
        ax.plot(epochs, history['train_' + metric], 'o-', label='Training (online)')
        ax.plot(epochs, history['val_' + metric], 'o-', label='Validation')
        ax.set(xlabel='Epoch', ylabel=label, xticks=list(epochs))
        ax.legend()
        ax.grid(alpha=0.25)
    axes[1].set_ylim(0, 1)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

@torch.no_grad()
def collect_predictions(model, loader, max_errors=8):
    model.eval()
    true_labels, predicted_labels, mistakes = [], [], []
    for images, labels in loader:
        predictions = model(images.to(DEVICE)).argmax(dim=1).cpu()
        true_labels.extend(labels.tolist())
        predicted_labels.extend(predictions.tolist())
        for idx in torch.where(predictions != labels)[0].tolist():
            if len(mistakes) < max_errors:
                mistakes.append((images[idx].clone(), int(labels[idx]), int(predictions[idx])))
    return np.asarray(true_labels), np.asarray(predicted_labels), mistakes

def show_errors(mistakes, class_names, mean, std):
    if not mistakes:
        print('No misclassified examples in this evaluation set.')
        return
    images = torch.stack([item[0] for item in mistakes])
    labels = torch.tensor([item[1] for item in mistakes])
    predictions = torch.tensor([item[2] for item in mistakes])
    show_images(images, labels, class_names, mean, std, predictions, n=len(mistakes))


In [ ]:
BASELINE_LR = 1e-3
baseline_run = run_experiment(CIFAR10CNN, BASELINE_LR, EPOCHS,
                              train_dataset, val_dataset, 'Baseline')
plot_history(baseline_run['history'], 'CIFAR-10 baseline')


### T3 · Your answer

1. What changes when `loss.backward()` runs? What changes when `optimizer.step()` runs?
2. Why do we use both `model.eval()` and `torch.no_grad()` for evaluation?

YOUR ANSWER (2–4 sentences total)


## T4 · Change one learning rate — 4 points · about 10 minutes

Choose **one** alternative Adam learning rate: `3e-4` or `3e-3`. State a hypothesis before running it. Change no other setting.
Use the provided driver so both models start from identical weights, have the same shuffle sequence, and receive the same two-epoch budget.

Complete the cell, run it once, compare the best validation accuracies and curves, and interpret the result. Improvement is **not** required for full credit.
One short run supports a limited observation; it does not prove that a learning rate is universally better.


### T4 · Hypothesis before running

**Chosen learning rate:** YOUR VALUE  
**Why I expect this change to help, and one possible drawback:** YOUR ANSWER


In [ ]:
# TODO T4: choose 3e-4 or 3e-3.
EXPERIMENT_LR = None

assert EXPERIMENT_LR in (3e-4, 3e-3), 'T4: choose one of the two specified learning rates.'
experiment_run = run_experiment(CIFAR10CNN, EXPERIMENT_LR, EPOCHS,
                                train_dataset, val_dataset, 'Learning-rate experiment')
assert all(torch.equal(baseline_run['initial_state'][k], experiment_run['initial_state'][k])
           for k in baseline_run['initial_state']), 'Initial weights differ.'
print('Initial weights match exactly.')
for run in [baseline_run, experiment_run]:
    print(f"{run['name']}: lr={run['lr']}, best validation={run['best_val_acc']:.3f}, epoch={run['best_epoch']}")
print('Validation difference (experiment - baseline):',
      round(experiment_run['best_val_acc'] - baseline_run['best_val_acc'], 4))
plot_history(experiment_run['history'], 'CIFAR-10 learning-rate experiment')
plt.figure(figsize=(6, 3.5))
for run in [baseline_run, experiment_run]:
    plt.plot(range(1, EPOCHS + 1), run['history']['val_acc'], 'o-', label=f"lr={run['lr']}")
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.title('One-variable comparison')
plt.xticks(range(1, EPOCHS + 1))
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


### T4 · Your comparison

**Best baseline validation accuracy / epoch:** YOUR VALUES  
**Best experiment validation accuracy / epoch:** YOUR VALUES  
**Difference (experiment minus baseline):** YOUR VALUE

What happened? What made the comparison fair, and what can this single short experiment not establish?

YOUR ANSWER (2–4 sentences)


## T5 · Test once and explain the mistakes — 3 points · about 10 minutes

The provided cell selects the higher-validation run; ties keep the baseline. Both returned models have their best validation weights restored.
Only now do we open a fixed **1,000-image test subset**. After seeing test results, do not change the model or its hyperparameters for this submission.

1. Create a **10×10 confusion matrix** with class names. Rows are true labels, columns are predictions. **(1 point)**
2. Display up to eight actual misclassified images with true and predicted labels; show all if fewer than eight exist. You may use `show_errors`. **(1 point)**
3. Identify a frequent off-diagonal confusion, explain one plausible cause, and say why test accuracy was not used to choose the model. **(1 point)**


In [ ]:
#@title Provided final model selection and test prediction collection
selected_run = (experiment_run if experiment_run['best_val_acc'] > baseline_run['best_val_acc']
                else baseline_run)
analysis_model = selected_run['model']
raw_test = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform)
test_idx = torch.randperm(len(raw_test), generator=torch.Generator().manual_seed(SEED + 1))[:N_TEST].tolist()
test_loader = make_loader(Subset(raw_test, test_idx))
y_true, y_pred, mistakes = collect_predictions(analysis_model, test_loader)
print('Selected by validation:', selected_run['name'], '| epoch:', selected_run['best_epoch'])
print(f'Final test-subset accuracy ({len(y_true)} images): {(y_true == y_pred).mean():.3f}')
print('Saved error examples:', len(mistakes))


In [ ]:
# TODO T5.1: build and display the confusion matrix.
# Hint: confusion_matrix(y_true, y_pred, labels=np.arange(10))
# Use class_names as display_labels and label the plot.

# TODO T5.2: show the saved mistakes.
# Hint: show_errors(mistakes, class_names, MEAN, STD)

# YOUR CODE HERE


### T5 · Your interpretation

**True class → predicted class / count:** YOUR VALUES  
**One plausible reason, using the error images:** YOUR ANSWER  
**Why we selected the model using validation rather than test accuracy:** YOUR ANSWER


## Submit to Moodle

1. Fill in your name, Neptun code, and any assistance used.
2. Allow time to restart the runtime and run the completed notebook from top to bottom. Fix unexpected errors. Provided TODO stops disappear when the tasks are complete.
3. Keep written answers, printed results, learning curves, the confusion matrix, and error images visible. Do not enable Colab's option to omit cell outputs when saving.
4. Rename the file to **`YOUR_NEPTUN_DL_Lab02.ipynb`** using your own code.
5. Choose **File → Download → Download .ipynb** and submit the downloaded file to Moodle **before the lab ends**. A sharing link alone is not sufficient.

Submit partial work on time if something remains incomplete; state what you tried and where you got stuck. Do not invent results.
If Moodle access is still unavailable, keep the downloaded file, tell the instructor, and email your work to the instructor as stated in the course README.


## References and credits

- [PyTorch: tensors and autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)
- [PyTorch: optimisation loop](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
- [PyTorch: saving the best model](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)
- [PyTorch: reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)
- [Fashion-MNIST: Xiao, Rasul and Vollgraf / Zalando Research](https://github.com/zalandoresearch/fashion-mnist)
- [CIFAR-10: Alex Krizhevsky / University of Toronto](https://www.cs.toronto.edu/~kriz/cifar.html)

Course material: Dr. Mohammed Salah Al-Radhi, BME. Dataset and framework credits remain with their respective authors. Follow the original terms when reusing third-party material.
